# Logistic Regression — Base vs Google Trends

**Part 1** trains and evaluates a logistic regression on price + engineered features.  
**Part 2** adds 5 Google Trends features and evaluates independently.  
**Part 3** compares both models head-to-head.

In [44]:
import pathlib
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    f1_score, log_loss, precision_recall_curve, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [45]:
ROOT            = pathlib.Path("../..")
DATA_DIR        = ROOT / "data"
ARTIFACTS_DIR   = pathlib.Path("artifacts")
PREDICTIONS_DIR = pathlib.Path("predictions")

NUMERIC_FEATURES = [
    "price_at_snapshot",
    "price_deviation_from_half",
    "days_before_close",
    "pct_lifetime_elapsed",
    "duration_days",
    "log_volume",
    "price_mean_7d",   "price_volatility_7d",  "price_min_7d",  "price_max_7d",
    "price_change_7d", "price_range_7d",        "price_trend_7d",
    "price_mean_14d",  "price_volatility_14d", "price_min_14d", "price_max_14d",
    "price_change_14d","price_range_14d",       "price_trend_14d",
]
TRENDS_FEATURES      = ["trend_value", "trend_ma4", "trend_change_4w", "trend_spike", "has_trend_data"]
CATEGORICAL_FEATURES = ["category"]
TARGET               = "outcome"

FEATURES_BASE   = NUMERIC_FEATURES + CATEGORICAL_FEATURES
FEATURES_TRENDS = NUMERIC_FEATURES + TRENDS_FEATURES + CATEGORICAL_FEATURES

---
## Load Data

The trends-enriched dataset contains all base features plus the 5 trend columns, split across two parquet files.

In [46]:
df = pd.read_parquet(DATA_DIR / "polymarket_ml_dataset_with_trends_clean.parquet")

df["category"] = df["category"].fillna("other")
df = df.dropna(subset=[TARGET])

train = df[df["split"] == "train"]
test  = df[df["split"] == "test"]

assert len(set(train["market_id"]) & set(test["market_id"])) == 0, "Market leakage detected"

y_train    = train[TARGET]
y_test     = test[TARGET]

counts           = train.groupby("market_id").size()
snapshot_weights = train["market_id"].map(counts).rdiv(1).values
class_weights    = compute_sample_weight("balanced", y_train)
sample_weights   = class_weights * snapshot_weights
sample_weights   = sample_weights / sample_weights.mean()
test_reset = test.reset_index(drop=True)

print(f"Total rows : {len(df):,}  |  Columns: {df.shape[1]}")
print(f"Train      : {len(train):,}  |  {train['market_id'].nunique():,} markets")
print(f"Test       : {len(test):,}   |  {test['market_id'].nunique():,} markets")
print(f"Trend coverage (has_trend_data=1): {df['has_trend_data'].mean():.1%}")
df.head()

Total rows : 1,448,142  |  Columns: 32
Train      : 1,159,652  |  16,774 markets
Test       : 288,490   |  4,174 markets
Trend coverage (has_trend_data=1): 99.6%


,market_id,snapshot_timestamp,days_before_close,pct_lifetime_elapsed,duration_days,price_at_snapshot,price_deviation_from_half,total_volume,log_volume,outcome,...,price_range_14d,price_trend_14d,split,category,question,trend_value,trend_ma4,trend_change_4w,trend_spike,has_trend_data
0,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-01 18:45:42.437000+00:00,59.22,0.1912,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000526,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
1,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-02 06:45:42.437000+00:00,58.72,0.1980,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000354,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
2,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-02 18:45:42.437000+00:00,58.22,0.2049,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000215,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
3,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-03 06:45:42.437000+00:00,57.72,0.2117,73,0.03,0.47,40175.18,10.601,0,...,0.03,0.000479,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
4,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-03 18:45:42.437000+00:00,57.22,0.2185,73,0.03,0.47,40175.18,10.601,0,...,0.03,0.000499,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1


---
## Shared Helpers

In [47]:
def build_pipeline(num_cols, cat_cols, C=1.0, solver="lbfgs", penalty="l2"):
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ("scaler",  StandardScaler()),
            ]), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ])),
        ("clf", LogisticRegression(
            max_iter=1000,
            solver=solver, C=C, penalty=penalty, random_state=42,
        )),
    ])

def get_threshold(y_true, y_prob):
    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    return float(thresh[np.argmax(f1)]), float(np.max(f1))

def evaluate(y_true, y_prob, label, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "AUC-ROC" : roc_auc_score(y_true, y_prob),
        "PR-AUC"  : average_precision_score(y_true, y_prob),
        "Log-loss": log_loss(y_true, y_prob),
        "Brier"   : brier_score_loss(y_true, y_prob),
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1"      : f1_score(y_true, y_pred),
    }
    print(f"\n{'─'*50}")
    print(f"  {label}  (threshold={threshold:.3f})")
    print(f"{'─'*50}")
    for name, val in metrics.items():
        print(f"  {name:<10}: {val:.4f}")
    print(f"{'─'*50}")
    return metrics

def per_category(test_df, y_prob, threshold):
    rows = []
    for cat in sorted(test_df["category"].unique()):
        mask = test_df["category"] == cat
        yt   = test_df.loc[mask, TARGET].values
        if len(np.unique(yt)) < 2 or len(yt) < 10:
            continue
        yp = y_prob[mask.values]
        rows.append({
            "category": cat,
            "n"       : int(mask.sum()),
            "AUC"     : roc_auc_score(yt, yp),
            "PR-AUC"  : average_precision_score(yt, yp),
            "F1"      : f1_score(yt, (yp >= threshold).astype(int)),
            "YES%"    : float(yt.mean()),
        })
    return pd.DataFrame(rows).set_index("category").sort_values("AUC", ascending=False)

def market_eval(test_df, y_prob, threshold):
    mdf = (
        test_df.assign(pred_prob=y_prob)
        .groupby("market_id")
        .agg(pred_prob=("pred_prob", "mean"), outcome=(TARGET, "first"))
        .reset_index()
    )
    mp, mt = mdf["pred_prob"].values, mdf["outcome"].values
    mpred  = (mp >= threshold).astype(int)
    print(f"Market-level evaluation ({len(mdf):,} markets)")
    print(f"  AUC-ROC  : {roc_auc_score(mt, mp):.4f}")
    print(f"  PR-AUC   : {average_precision_score(mt, mp):.4f}")
    print(f"  Brier    : {brier_score_loss(mt, mp):.4f}")
    print(f"  Accuracy : {accuracy_score(mt, mpred):.4f}")
    print(f"  F1       : {f1_score(mt, mpred):.4f}")
    return {"AUC-ROC": roc_auc_score(mt,mp), "PR-AUC": average_precision_score(mt,mp),
            "Brier": brier_score_loss(mt,mp), "Accuracy": accuracy_score(mt,mpred), "F1": f1_score(mt,mpred)}

---
## Grid Search — Hyperparameter Tuning

Search over `C` (regularization strength) and `penalty` (L1 vs L2) using 5-fold CV scored by AUC-ROC.
Best params are used for both Base and Trends models.

In [48]:
param_grid = [
    {"clf__C": [0.01, 0.1, 1.0, 10.0, 100.0], "clf__penalty": ["l2"], "clf__solver": ["lbfgs"]},
    {"clf__C": [0.01, 0.1, 1.0, 10.0, 100.0], "clf__penalty": ["l1"], "clf__solver": ["liblinear"]},
]
_pipe_gs = build_pipeline(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
grid = GridSearchCV(_pipe_gs, param_grid, scoring="roc_auc", cv=5, n_jobs=-1, verbose=1)
grid.fit(train[FEATURES_BASE], y_train, clf__sample_weight=sample_weights)

best_C       = grid.best_params_["clf__C"]
best_solver  = grid.best_params_["clf__solver"]
best_penalty = grid.best_params_["clf__penalty"]
print(f"\nBest params : C={best_C}  penalty={best_penalty}  solver={best_solver}")
print(f"Best CV AUC : {grid.best_score_:.4f}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits

Best params : C=0.01  penalty=l2  solver=lbfgs
Best CV AUC : 0.8664


---
## Baseline — Market Price

The simplest predictor: use `price_at_snapshot` directly as the probability.
This is the crowd's consensus — a useful sanity check for any model we build.

In [49]:
y_prob_baseline        = test_reset["price_at_snapshot"].values
thresh_bl, f1_bl       = get_threshold(y_test, y_prob_baseline)
print(f"Baseline threshold: {thresh_bl:.3f}  |  F1: {f1_bl:.4f}")

metrics_baseline_row = evaluate(y_test, y_prob_baseline, "Baseline (market price)", thresh_bl)
metrics_baseline_mkt = market_eval(test_reset, y_prob_baseline, thresh_bl)
cat_baseline         = per_category(test_reset, y_prob_baseline, thresh_bl)

Baseline threshold: 0.500  |  F1: 0.6699

──────────────────────────────────────────────────
  Baseline (market price)  (threshold=0.500)
──────────────────────────────────────────────────
  AUC-ROC   : 0.8890
  PR-AUC    : 0.7440
  Log-loss  : 0.3278
  Brier     : 0.1004
  Accuracy  : 0.8703
  F1        : 0.6699
──────────────────────────────────────────────────
Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9420
  PR-AUC   : 0.8283
  Brier    : 0.0652
  Accuracy : 0.9178
  F1       : 0.7351


pipe_base = build_pipeline(
    num_cols=NUMERIC_FEATURES,
    cat_cols=CATEGORICAL_FEATURES,
    C=best_C, solver=best_solver, penalty=best_penalty,
)
pipe_base.fit(train[FEATURES_BASE], y_train, clf__sample_weight=sample_weights)

y_prob_base          = pipe_base.predict_proba(test[FEATURES_BASE])[:, 1]
thresh_base, f1_base = get_threshold(y_test, y_prob_base)
print(f"Optimal threshold: {thresh_base:.3f}  |  F1: {f1_base:.4f}")

## 1.1 Train

In [50]:
pipe_base = build_pipeline(
    num_cols=NUMERIC_FEATURES,
    cat_cols=CATEGORICAL_FEATURES,
)
pipe_base.fit(train[FEATURES_BASE], y_train, clf__sample_weight=sample_weights)

y_prob_base        = pipe_base.predict_proba(test[FEATURES_BASE])[:, 1]
thresh_base, f1_base = get_threshold(y_test, y_prob_base)
print(f"Optimal threshold: {thresh_base:.3f}  |  F1: {f1_base:.4f}")

Optimal threshold: 0.681  |  F1: 0.6661


## 1.2 Row-Level Evaluation

In [51]:
metrics_base_row = evaluate(y_test, y_prob_base, "Base Model", thresh_base)


──────────────────────────────────────────────────
  Base Model  (threshold=0.681)
──────────────────────────────────────────────────
  AUC-ROC   : 0.8881
  PR-AUC    : 0.7231
  Log-loss  : 0.4105
  Brier     : 0.1255
  Accuracy  : 0.8589
  F1        : 0.6661
──────────────────────────────────────────────────


## 1.3 Per-Category Breakdown

In [52]:
cat_base = per_category(test_reset, y_prob_base, thresh_base)
cat_base.style.format({"AUC": "{:.4f}", "PR-AUC": "{:.4f}", "F1": "{:.4f}", "YES%": "{:.1%}"})

,n,AUC,PR-AUC,F1,YES%
category,,,,,
geopolitics,16578,0.9357,0.7431,0.6693,13.8%
entertainment,37027,0.9189,0.7270,0.6498,14.9%
finance,25093,0.9174,0.8090,0.7098,24.3%
politics_global,22591,0.9076,0.7405,0.6784,22.3%
politics_us,52699,0.9053,0.7671,0.7305,24.7%
crypto,24586,0.9042,0.7315,0.7173,26.1%
science_tech,15530,0.8857,0.7157,0.6587,17.8%
other,1364,0.8692,0.3100,0.4545,4.7%
sports,93022,0.8428,0.6627,0.5838,20.9%


## 1.4 Market-Level Evaluation

In [53]:
metrics_base_mkt = market_eval(test_reset, y_prob_base, thresh_base)

Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9382
  PR-AUC   : 0.8086
  Brier    : 0.0787
  Accuracy : 0.9118
  F1       : 0.7266


## 1.5 Feature Importance

In [54]:
clf_step  = pipe_base.named_steps["clf"]
prep_step = pipe_base.named_steps["preprocessor"]
cat_names = list(prep_step.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
all_names = NUMERIC_FEATURES + cat_names

importance_df_base = (
    pd.DataFrame({"feature": all_names, "coefficient": clf_step.coef_[0]})
    .assign(abs_coef=lambda d: d["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
    .reset_index(drop=True)
)

importance_df_base.head(20).style.bar(
    subset=["coefficient"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,feature,coefficient
0,price_at_snapshot,1.017008
1,log_volume,0.465960
2,duration_days,-0.454654
3,price_max_14d,0.430810
4,days_before_close,0.412575
5,price_min_14d,0.382790
6,category_geopolitics,-0.318058
7,category_other,0.293469
8,category_sports,-0.267823
9,price_range_14d,0.195304


pipe_trends = build_pipeline(
    num_cols=NUMERIC_FEATURES + TRENDS_FEATURES,
    cat_cols=CATEGORICAL_FEATURES,
    C=best_C, solver=best_solver, penalty=best_penalty,
)
pipe_trends.fit(train[FEATURES_TRENDS], y_train, clf__sample_weight=sample_weights)

y_prob_trends            = pipe_trends.predict_proba(test[FEATURES_TRENDS])[:, 1]
thresh_trends, f1_trends = get_threshold(y_test, y_prob_trends)
print(f"Optimal threshold: {thresh_trends:.3f}  |  F1: {f1_trends:.4f}")

## 2.1 Train

In [55]:
pipe_trends = build_pipeline(
    num_cols=NUMERIC_FEATURES + TRENDS_FEATURES,
    cat_cols=CATEGORICAL_FEATURES,
)
pipe_trends.fit(train[FEATURES_TRENDS], y_train, clf__sample_weight=sample_weights)

y_prob_trends          = pipe_trends.predict_proba(test[FEATURES_TRENDS])[:, 1]
thresh_trends, f1_trends = get_threshold(y_test, y_prob_trends)
print(f"Optimal threshold: {thresh_trends:.3f}  |  F1: {f1_trends:.4f}")

Optimal threshold: 0.682  |  F1: 0.6656


## 2.2 Row-Level Evaluation

In [56]:
metrics_trends_row = evaluate(y_test, y_prob_trends, "Trends Model", thresh_trends)


──────────────────────────────────────────────────
  Trends Model  (threshold=0.682)
──────────────────────────────────────────────────
  AUC-ROC   : 0.8882
  PR-AUC    : 0.7227
  Log-loss  : 0.4101
  Brier     : 0.1254
  Accuracy  : 0.8587
  F1        : 0.6656
──────────────────────────────────────────────────


## 2.3 Per-Category Breakdown

In [57]:
cat_trends = per_category(test_reset, y_prob_trends, thresh_trends)
cat_trends.style.format({"AUC": "{:.4f}", "PR-AUC": "{:.4f}", "F1": "{:.4f}", "YES%": "{:.1%}"})

,n,AUC,PR-AUC,F1,YES%
category,,,,,
geopolitics,16578,0.9356,0.7414,0.6702,13.8%
entertainment,37027,0.9190,0.7275,0.6490,14.9%
finance,25093,0.9173,0.8091,0.7084,24.3%
crypto,24586,0.9069,0.7377,0.7154,26.1%
politics_global,22591,0.9063,0.7329,0.6814,22.3%
politics_us,52699,0.9049,0.7676,0.7317,24.7%
science_tech,15530,0.8858,0.7130,0.6537,17.8%
other,1364,0.8688,0.3097,0.4386,4.7%
sports,93022,0.8426,0.6624,0.5832,20.9%


## 2.4 Market-Level Evaluation

In [58]:
metrics_trends_mkt = market_eval(test_reset, y_prob_trends, thresh_trends)

Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9382
  PR-AUC   : 0.8085
  Brier    : 0.0786
  Accuracy : 0.9116
  F1       : 0.7257


## 2.5 Feature Importance

Trend features are highlighted in yellow — their magnitude relative to price features shows how much signal they add.

In [59]:
clf_step  = pipe_trends.named_steps["clf"]
prep_step = pipe_trends.named_steps["preprocessor"]
cat_names = list(prep_step.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
all_names = NUMERIC_FEATURES + TRENDS_FEATURES + cat_names

importance_df = (
    pd.DataFrame({"feature": all_names, "coefficient": clf_step.coef_[0]})
    .assign(
        abs_coef = lambda d: d["coefficient"].abs(),
        is_trend = lambda d: d["feature"].isin(TRENDS_FEATURES),
    )
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
    .reset_index(drop=True)
)

print("Trend feature coefficients:")
print(importance_df[importance_df["is_trend"]].to_string(index=False))
print()

importance_df.head(25).style.bar(
    subset=["coefficient"], align="zero", color=["#d65f5f", "#5fba7d"]
).apply(
    lambda col: ["background-color: #fff3cd" if v else "" for v in importance_df.head(25)["is_trend"]],
    axis=0, subset=["feature", "coefficient"]
)

Trend feature coefficients:
        feature  coefficient  is_trend
    trend_value     0.335576      True
      trend_ma4    -0.245592      True
trend_change_4w    -0.105540      True
    trend_spike     0.036114      True
 has_trend_data    -0.033281      True



,feature,coefficient,is_trend
0,price_at_snapshot,1.027565,False
1,log_volume,0.471123,False
2,duration_days,-0.451471,False
3,price_max_14d,0.437380,False
4,days_before_close,0.411406,False
5,price_min_14d,0.389733,False
6,trend_value,0.335576,True
7,trend_ma4,-0.245592,True
8,category_sports,-0.239555,False
9,category_geopolitics,-0.233098,False


---
# Part 3 — Comparison

Head-to-head: Base vs Trends across row-level metrics, market-level metrics, and per-category AUC.

## 3.1 Row-Level

In [60]:
row_comparison = pd.DataFrame({
    "Baseline": metrics_baseline_row,
    "Base":     metrics_base_row,
    "Trends":   metrics_trends_row,
})
row_comparison["Δ Base"]   = row_comparison["Base"]   - row_comparison["Baseline"]
row_comparison["Δ Trends"] = row_comparison["Trends"] - row_comparison["Baseline"]
row_comparison.style.format("{:.4f}").bar(
    subset=["Δ Base", "Δ Trends"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,Baseline,Base,Trends,Δ Base,Δ Trends
AUC-ROC,0.8890,0.8881,0.8882,-0.0009,-0.0008
PR-AUC,0.7440,0.7231,0.7227,-0.0209,-0.0213
Log-loss,0.3278,0.4105,0.4101,0.0827,0.0823
Brier,0.1004,0.1255,0.1254,0.0251,0.0249
Accuracy,0.8703,0.8589,0.8587,-0.0114,-0.0116
F1,0.6699,0.6661,0.6656,-0.0038,-0.0043


## 3.2 Market-Level

In [61]:
mkt_comparison = pd.DataFrame({
    "Baseline": metrics_baseline_mkt,
    "Base":     metrics_base_mkt,
    "Trends":   metrics_trends_mkt,
})
mkt_comparison["Δ Base"]   = mkt_comparison["Base"]   - mkt_comparison["Baseline"]
mkt_comparison["Δ Trends"] = mkt_comparison["Trends"] - mkt_comparison["Baseline"]
mkt_comparison.style.format("{:.4f}").bar(
    subset=["Δ Base", "Δ Trends"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,Baseline,Base,Trends,Δ Base,Δ Trends
AUC-ROC,0.9420,0.9382,0.9382,-0.0038,-0.0038
PR-AUC,0.8283,0.8086,0.8085,-0.0197,-0.0198
Brier,0.0652,0.0787,0.0786,0.0135,0.0135
Accuracy,0.9178,0.9118,0.9116,-0.0060,-0.0062
F1,0.7351,0.7266,0.7257,-0.0085,-0.0095


## 3.3 Per-Category AUC Delta

In [62]:
cat_comparison = cat_baseline[["n", "AUC", "YES%"]].rename(columns={"AUC": "AUC (baseline)"})
cat_comparison["AUC (base)"]   = cat_base["AUC"]
cat_comparison["AUC (trends)"] = cat_trends["AUC"]
cat_comparison["Δ Base"]       = cat_comparison["AUC (base)"]   - cat_comparison["AUC (baseline)"]
cat_comparison["Δ Trends"]     = cat_comparison["AUC (trends)"] - cat_comparison["AUC (baseline)"]
(
    cat_comparison
    .sort_values("AUC (baseline)", ascending=False)
    .style
    .format("{:.4f}", subset=["AUC (baseline)", "AUC (base)", "AUC (trends)", "Δ Base", "Δ Trends"])
    .format("{:.1%}", subset=["YES%"])
    .bar(subset=["Δ Base", "Δ Trends"], align="zero", color=["#d65f5f", "#5fba7d"])
)

,n,AUC (baseline),YES%,AUC (base),AUC (trends),Δ Base,Δ Trends
category,,,,,,,
geopolitics,16578,0.9265,13.8%,0.9357,0.9356,0.0092,0.0091
finance,25093,0.9183,24.3%,0.9174,0.9173,-0.0009,-0.0010
entertainment,37027,0.9101,14.9%,0.9189,0.9190,0.0088,0.0089
politics_us,52699,0.9083,24.7%,0.9053,0.9049,-0.0030,-0.0034
politics_global,22591,0.9057,22.3%,0.9076,0.9063,0.0019,0.0006
crypto,24586,0.9052,26.1%,0.9042,0.9069,-0.0010,0.0017
science_tech,15530,0.8876,17.8%,0.8857,0.8858,-0.0019,-0.0018
other,1364,0.8847,4.7%,0.8692,0.8688,-0.0154,-0.0159
sports,93022,0.8400,20.9%,0.8428,0.8426,0.0028,0.0026


---
## Save Artifacts

In [63]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": pipe_base},   ARTIFACTS_DIR / "model_base.joblib")
joblib.dump({"pipeline": pipe_trends}, ARTIFACTS_DIR / "model_trends.joblib")
print(f"Models saved → {ARTIFACTS_DIR}")

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
pred_df = test[["market_id", "category", TARGET]].copy().reset_index(drop=True)
pred_df["pred_prob_base"]    = y_prob_base
pred_df["pred_prob_trends"]  = y_prob_trends
pred_df["pred_label_base"]   = (y_prob_base   >= thresh_base).astype(int)
pred_df["pred_label_trends"] = (y_prob_trends >= thresh_trends).astype(int)
preds_path = PREDICTIONS_DIR / "predictions.csv"
pred_df.to_csv(preds_path, index=False)
print(f"Predictions saved → {preds_path}")
pred_df.head()

Models saved → artifacts
Predictions saved → predictions/predictions.csv


,market_id,category,outcome,pred_prob_base,pred_prob_trends,pred_label_base,pred_label_trends
0,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.696539,0.680981,1,0
1,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.768445,0.756340,1,1
2,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.775166,0.761309,1,1
3,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.774722,0.761736,1,1
4,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.883920,0.876014,1,1
